<a href="https://colab.research.google.com/github/12halima/Transport_Recommander/blob/main/Process_GTFS-OSM/extract_BoudingBox.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import pandas as pd

def extract_bounding_boxes(root_folder):
    """
    Parcourt tous les sous-dossiers du root_folder.
    Pour chaque dossier contenant stops.txt :
        - lit les coordonnées
        - calcule min/max lat/lon
        - ajoute une ligne dans gtfs_bounding_boxes.csv
    """

    output_csv = os.path.join(root_folder, "gtfs_bounding_boxes.csv")
    results = []

    for dirpath, _, filenames in os.walk(root_folder):
        if "stops.txt" in filenames:
            stops_path = os.path.join(dirpath, "stops.txt")

            try:
                df = pd.read_csv(stops_path, dtype=str, encoding="utf-8")

                df["stop_lat"] = df["stop_lat"].astype(float)
                df["stop_lon"] = df["stop_lon"].astype(float)

                min_lat = df["stop_lat"].min()
                max_lat = df["stop_lat"].max()
                min_lon = df["stop_lon"].min()
                max_lon = df["stop_lon"].max()

                # uniquement le nom du sous-dossier
                folder_name = os.path.basename(dirpath)

                results.append({
                    "source_folder": folder_name,
                    "min_lat": min_lat,
                    "max_lat": max_lat,
                    "min_lon": min_lon,
                    "max_lon": max_lon
                })

            except Exception as e:
                print(f"Erreur lecture {stops_path}: {e}")

    # dataframe final
    result_df = pd.DataFrame(results)

    # sauvegarde
    result_df.to_csv(output_csv, index=False, encoding="utf-8")

    print("Fichier résumé créé dans GTFS_CLEAN :", output_csv)
    return result_df


# Exécution
root_folder = "/content/drive/MyDrive/GTFS_CLEAN"
df_bbox = extract_bounding_boxes(root_folder)

df_bbox


Fichier résumé créé dans GTFS_CLEAN : /content/drive/MyDrive/GTFS_CLEAN/gtfs_bounding_boxes.csv


,source_folder,min_lat,max_lat,min_lon,max_lon
0,Vectalia Movilidad (bus de la ville de Cáceres),39.196675,39.503281,-6.422665,-6.330012
1,Xunta de Galicia Buses,41.814355,43.742009,-9.271771,-6.806418
2,Àrea Metropolitana de Barcelona (AMB),41.262519,41.527356,1.846658,2.289360
3,Viagón Coaches,40.000000,41.218444,-6.690278,-5.000000
4,TUSSAM (Seville bus and tram),37.310978,37.451149,-6.013968,-5.847688
...,...,...,...,...,...
100,Ancebus,40.608600,41.269200,-6.747800,-5.411400
101,Alvarez Travelers Coaches,41.513889,42.171861,-7.004722,-5.739167
102,Alavabus,42.457600,43.260200,-3.240800,-1.645300
103,ALSA buses,27.836208,43.629317,-15.665996,2.195058


In [4]:
import pandas as pd

def show_invalid_bboxes(csv_path):
    """
    Lit le fichier gtfs_bounding_boxes.csv et affiche les lignes où :
        - min_lat >= max_lat
        - min_lon >= max_lon
    """
    df = pd.read_csv(csv_path, encoding="utf-8")

    invalid = df[
        (df["min_lat"] >= df["max_lat"]) |
        (df["min_lon"] >= df["max_lon"])
    ]

    if invalid.empty:
        print("Aucune ligne invalide. Les min/max sont cohérents.")
    else:
        print("Lignes invalides détectées :")
        print(invalid)

    return invalid

# Exemple d'appel
csv_path = "/content/drive/MyDrive/GTFS_CLEAN/gtfs_bounding_boxes.csv"
invalid_rows = show_invalid_bboxes(csv_path)


Aucune ligne invalide. Les min/max sont cohérents.
